In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"
from functools import partial
import time
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt


import matplotlib as mpl
from matplotlib import rc
rc('font',**{'family':'serif','serif':['Helvetica']})
mpl.rcParams['text.usetex'] = True
mpl.rcParams.update({'font.size': 10})
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}\usepackage{amsmath}\usepackage{upgreek}"

In [ ]:
import jax
import jax.numpy as jnp
# jax.config.update("jax_enable_x64", True)
gpus = jax.devices()
print(gpus)

jax.config.update("jax_default_device", gpus[0])

import diffrax
import equinox as eqx
import optax

from haiku import PRNGSequence

from dmpe.data_management import DataPaths
from dmpe.evaluation.experiment_utils import get_experiment_ids, load_experiment_results
from dmpe.models.models import NeuralEulerODECartpole

---

In [ ]:
from dmpe.utils.env_utils.fluid_tank_utils import setup_env as setup_fluid_tank_env

In [ ]:
env, _ = setup_fluid_tank_env()

In [ ]:
def recursive_feasibility_fluid_tank(h_k, u_k, tau, h_max, u_max, static_params):
    valid_state = jnp.logical_and(h_k <= h_max, h_k >= 0)
    valid_action = jnp.logical_and(u_k <= u_max, u_k >=0)

    valid = jnp.logical_and(valid_state, valid_action)

    valid_action_next_step_upper = (
        u_k <= (
            static_params.base_area / tau * (h_max - h_k) 
            + static_params.c_d * static_params.orifice_area * jnp.sqrt(2 * static_params.g * h_k)
        )
    )

    valid_action_next_step = jnp.logical_and(valid_action_next_step_upper, u_k >= 0)

    valid = jnp.logical_and(valid, valid_action_next_step)

    return valid

In [ ]:
h_max=env.env_properties.physical_normalizations.height.max
u_max=env.env_properties.action_normalizations.inflow.max

In [ ]:
recursive_feasibility_parameterized = eqx.filter_jit(
    partial(
        recursive_feasibility_fluid_tank,
        tau=env.tau,
        h_max=h_max,
        u_max=u_max,
        static_params=env.env_properties.static_params
    )
)
recursive_feasibility_parameterized

In [ ]:
heights_ = jnp.linspace(0, h_max, 250)
inflows_ = jnp.linspace(0, u_max, 250)

static_params = env.env_properties.static_params
allowed_inflow = (
    static_params.base_area / env.tau * (h_max - heights_) 
    + static_params.c_d * static_params.orifice_area * jnp.sqrt(2 * static_params.g * heights_)
)
allowed_inflow = jnp.clip(allowed_inflow, 0, 0.2)

heights, inflows = jnp.meshgrid(heights_, inflows_)

out_bool = jax.vmap(jax.vmap(recursive_feasibility_parameterized))(heights, inflows)

fig, ax = plt.subplots(1,1, figsize=(10,10), sharey=True)

ax.imshow(out_bool, cmap="plasma", extent=[0, h_max, u_max, 0], interpolation="nearest")

ax.plot(heights_, allowed_inflow)
ax.set_xlabel(r"$h$")
ax.set_ylabel(r"$q$")

fig.tight_layout()
plt.show()

## Evaluate recursive feasibility of experiment data:

In [ ]:
from dmpe.evaluation.experiment_utils import get_experiment_ids, load_experiment_results, load_all_experiment_results

cart_pole_data_path = DataPaths().cs_experiments / "dmpe" / "fluid_tank"
dmpe_experiment_results = load_all_experiment_results(cart_pole_data_path, model_class=None)

In [ ]:
observations, norm_actions = dmpe_experiment_results[0]["observations"], dmpe_experiment_results[0]["actions"]
states = env.vmap_generate_state_from_observation(observations)
actions = env.env_properties.action_normalizations.inflow.denormalize(norm_actions)

In [ ]:
heights_ = jnp.linspace(0, h_max, 250)
inflows_ = jnp.linspace(0, u_max, 250)

static_params = env.env_properties.static_params
allowed_inflow = (
    static_params.base_area / env.tau * (h_max - heights_) 
    + static_params.c_d * static_params.orifice_area * jnp.sqrt(2 * static_params.g * heights_)
)
allowed_inflow = jnp.clip(allowed_inflow, 0, 0.2)


heights, inflows = jnp.meshgrid(heights_, inflows_)

out_bool = jax.vmap(jax.vmap(recursive_feasibility_parameterized))(heights, inflows)

####


fig, ax = plt.subplots(1,1, figsize=(10,10), sharey=True)
ax.imshow(out_bool, cmap="plasma", extent=[0, h_max, u_max, 0], interpolation="nearest", aspect='auto')
ax.invert_yaxis()

ax.plot(heights_, allowed_inflow)
ax.set_xlabel(r"$h$")
ax.set_ylabel(r"$q$")

ax.scatter(states.physical_state.height[:-1], actions, s=1, c="r")

fig.tight_layout()
plt.show()